# Pulling Data From OMOP Tables

In [ ]:
import pandas as pd
import os
import numpy as np
import json
import subprocess
from datetime import *

Getting Data for Breast Cancer Events

In [ ]:
dataset = %env WORKSPACE_CDR
dataset

query = f"""
SELECT person_id, condition_start_datetime, condition_source_value, source_concept_name as icd_description
FROM `{dataset}.ds_condition_occurrence` co
WHERE co.PERSON_ID IN (
    SELECT
        distinct person_id
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
    WHERE
        cb_search_person.person_id IN (
            SELECT
                person_id
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
        )
    )
    AND (UPPER(source_concept_name) LIKE '%MALIGNANT%' AND UPPER(source_concept_name) LIKE '%NEOPLASM%')
    AND (condition_source_value LIKE 'C50%' OR condition_source_value LIKE '174%')
"""

malignant_bc_data = pd.read_gbq(query,
                        dialect = "standard",
                        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
                        progress_bar_type="tqdm_notebook")

malignant_bc_data.head()

Sorting Malignant BC Data by Person ID, and by Event Date

In [ ]:
cancer_sorted = malignant_bc_data.sort_values(by='person_id').groupby('person_id', group_keys=False).apply(lambda x: x.sort_values(by='condition_start_datetime'))
cancer_sorted = cancer_sorted.reset_index(drop=True)
cancer_sorted.head()

Finding First and Last Event for Each Person

In [ ]:
minmax_cancer_data = cancer_sorted.groupby(['person_id']).agg({'condition_start_datetime': [np.min,np.max]})
minmax_cancer_data.columns = minmax_cancer_data.columns.get_level_values(1)
minmax_cancer_data = minmax_cancer_data.reset_index()
minmax_cancer_data.head()

Removing People With only One BC Event

In [ ]:
new_minmax_data = pd.DataFrame()
for index, row in minmax_cancer_data.iterrows():
    if row['amin'] != row['amax']:
        row_to_extract = minmax_cancer_data.iloc[index]
        new_minmax_data = pd.concat([new_minmax_data, pd.DataFrame([row_to_extract])])
new_minmax_data = new_minmax_data.reset_index(drop=True)
new_minmax_data.head()

Getting Data & Concept Codes for All EHR Events

In [ ]:
# Required specs to run this block: 16 CPUs 60 Gigs of RAM

dataset = %env WORKSPACE_CDR
dataset

query = f"""
SELECT
    person_id,
    condition_start_datetime,
    condition_source_value,
    source_concept_name as icd_description,
    concept_code,
    c.concept_id
FROM
    `{dataset}.ds_condition_occurrence` co
LEFT JOIN
    `{dataset}.concept` c ON co.condition_concept_id = c.concept_id
WHERE co.PERSON_ID IN
    (
        SELECT
            distinct person_id
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
        WHERE
            cb_search_person.person_id IN

            (
                SELECT
                    person_id
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
            )
    )
"""

all_codes = pd.read_gbq(query,
                        dialect = "standard",
                        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
                        progress_bar_type="tqdm_notebook")

# all_codes.head()

Finding Overlaps Between All Concept Codes and new_minmax_data (All People Have at Least 2 BC Events)

In [ ]:
verified_bc_patients = pd.DataFrame()
shared_data = all_codes['person_id'].isin(new_minmax_data['person_id'])
verified_bc_patients = all_codes[shared_data]
verified_bc_patients.head()

Sorting Cancer Patients of Interest by person_id and Event Date

In [ ]:
sorted_bc_patients = verified_bc_patients.sort_values(by='person_id').groupby('person_id', group_keys=False).apply(lambda x: x.sort_values(by='condition_start_datetime'))
sorted_bc_patients = sorted_bc_patients.reset_index(drop=True)
sorted_bc_patients.head()

Adds a day_diff Column to track time from BC Diagnosis to Each Event (Before BC Diagnosis)

In [ ]:
amin_mapping = new_minmax_data.groupby('person_id')['amin'].first()

sorted_bc_patients['day0_relative_time'] = sorted_bc_patients['person_id'].map(amin_mapping)

sorted_bc_patients['day_diff'] = (sorted_bc_patients['condition_start_datetime'] - sorted_bc_patients['day0_relative_time']).dt.days
del sorted_bc_patients['day0_relative_time']

pre_bc_events = sorted_bc_patients[sorted_bc_patients['day_diff'] <= 0]

pre_bc_events.head()

Making 100 Day Bins from -15000 Days to 0 Days Before BC Diagnosis

In [ ]:
bins = np.arange(-15000, 100, 100)
labels = [f"{bins[i]},{bins[i+1]}" for i in range(len(bins)-1)]

Applying Bins to Dataframe and Removing NAs

In [ ]:
pre_bc_events['bin'] = pd.cut(pre_bc_events['day_diff'],labels=labels,bins=bins)
binned_cancer_data = pre_bc_events
binned_individuals = binned_cancer_data[pd.notna(binned_cancer_data['bin'])]
binned_individuals.head()

Saves binned_individuals to Workplace Bucket as binned_individuals.csv

In [ ]:
# This snippet assumes you run setup first
# TAKES A WHILE (10-15 minutes) but as as long as the disk is not deleted we have complete access to our pre-filtered data.
# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
binned_individuals['condition_start_datetime'] = binned_individuals['condition_start_datetime'].astype(str)

my_dataframe = binned_individuals

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'binned_individuals.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

Gets a List of JSON String Dicts for Each Unique person_id

In [ ]:
groupedDataFrame = binned_individuals.groupby('person_id')
dfArray = [json.dumps(group.reset_index(drop=True).to_dict()) for _, group in groupedDataFrame]

unique_personIDs = list(binned_individuals['person_id'].unique())

Writing Data to File

In [ ]:
for i in range(len(unique_personIDs)):
    my_dataframe = binned_individuals
    destination_filename = f"input{i}.txt"
    with open(destination_filename, 'w') as output:
        output.write(dfArray[i])
my_bucket = os.getenv('WORKSPACE_BUCKET')
args = ["gsutil","-m", "cp", "./input*.txt", f"{my_bucket}/data/"]
subprocess.run(args, capture_output=False)